In [1]:
import os
from google.colab import userdata


# Définissez vos informations de manière sécurisée
GITHUB_USER = "koubra-gaby"
#GITHUB_TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

# Configuration de l'URL avec authentification
REPO_URL = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Cloner le dépôt si ce n'est pas déjà fait
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("Dépôt cloné avec succès !")
else:
    print("Le dépôt existe déjà.")


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 381, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 381 (delta 14), reused 9 (delta 3), pack-reused 348 (from 1)
Receiving objects: 100% (381/381), 22.53 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (178/178), done.
Dépôt cloné avec succès !


## training fastext model for chadian arabic corpus

In [2]:
%%capture
!pip install gensim
!pip install nltk


In [3]:
## Importing libraries
from gensim.models import FastText
from gensim.test.utils import common_texts

In [4]:
import re
from nltk.tokenize import word_tokenize


def preparer_sentences_entrainer_fastext(fichier1, fichier2, file_to_save):
    sentences = []

    for fichier in [fichier1, fichier2]:
        with open(fichier, "r", encoding="utf-8-sig") as f:
            for ligne in f:
                ligne = ligne.strip()

                if ligne:
                    # Remplacer les ponctuations par des espaces,
                    # sauf les apostrophes et les tirets
                    ligne = re.sub(
                        r"[^\w\s'-]",
                        " ",
                        ligne,
                        flags=re.UNICODE
                    )

                    # Tokenisation mot par mot
                    tokens = word_tokenize(
                        ligne,
                        preserve_line=True
                    )

                    if tokens:
                        sentences.append(tokens)

      ## training fastext
    model = FastText(
      sentences,
      vector_size=1024,
      window=5,
      min_count=1,
      epochs=10,
      workers=4,
      sg=1,
  )
    model.wv.save_word2vec_format(file_to_save)

    print("Model trained and savec successfully")

    return model,sentences

In [ ]:
## Loading the corpus then tokenizing it
data_path = "/content/Chadian_arabic-update/Data/full_dataset/"
model_arabic,sentences_arabic = preparer_sentences_entrainer_fastext(data_path + "arabe_tchad3.txt",data_path + "arabic_not_bible.txt", data_path + "arabic_embedding.vec")
sentences_arabic

In [ ]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_arabic.wv.most_similar(
    "malak",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

In [ ]:
model_fench,sentences_french = preparer_sentences_entrainer_fastext(data_path + "V_francais_3.txt",data_path + "french_not_bible.txt", data_path + "fench_embedding.vec")
sentences_french

In [ ]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_fench.wv.most_similar(
    "bonjour",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

In [ ]:
!git clone https://github.com/facebookresearch/MUSE.git

In [ ]:
!python MUSE/unsupervised.py --src_lang fr --tgt_lang shu --src_emb data_path/fench_embedding.vec --tgt_emb data_path/arabic_embedding.vec --n_refinement 5
